# Modeling pipeline — eval harness, causal baseline, feature builder

Three reusable pieces, each validated before the next:
1. **Eval harness** — Time-Stratified AUC (TS-AUC), the competition metric, over the online rows.
2. **Causal baseline** — a value-based online detector; the number any model must beat.
3. **Feature builder** — per-timestep causal features for a model.

Everything is **causal**: at online time `t` it uses only the reference segment (`period==1`) plus online values up to `t` (no look-ahead), matching the sequential-inference constraint. See `investigate.ipynb` for the dataset structure.

In [20]:
import numpy as np
import pandas as pd

## Step 1 — Local eval harness (Time-Stratified AUC)

The competition metric is **TS-AUC**, *not* a globally pooled AUC. At each online step `t` (the `t`-th online observation, aligned across series), it computes a **cross-sectional AUC across the series alive at step `t`**:
- a series is **positive** at `t` if its break has already occurred (`target == 1`), else **negative**;
- `AUC(t)` ranks those series' scores against each other *at that step*;
- TS-AUC = average of `AUC(t)` weighted by pair count `w(t) = n_pos(t) · n_neg(t)`.

**Key consequence:** every comparison is *within a fixed step*, so a score that doesn't depend on series content **cannot beat 0.5** — e.g. `n_online` (identical for all series at step `t`) scores exactly 0.5. This is the opposite of pooled AUC and it's the point: you only score by telling series apart *at the same moment*. We keep `pooled_auc` for diagnostics only.

**Intuition:** TS-AUC rewards ranking series *against each other at the same step*, so a good score must reflect **this series' deviation from its own reference**, on a scale comparable across series. Anything shared by all series at a step (like elapsed time) cancels out.

In [ ]:
def _auc_pair_stat(scores, y):
    """Rank-sum within one group -> (concordant_pairs, n_pos*n_neg); ties count 0.5.
    (concordant / pairs) is that group's AUC; returns (0, 0) if a class is absent."""
    y = np.asarray(y).astype(int); scores = np.asarray(scores, dtype=float)
    n_pos = int(y.sum()); n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.0, 0.0
    order = np.argsort(scores, kind="mergesort"); s = scores[order]
    ranks = np.empty(len(scores)); i = 0
    while i < len(s):                                   # average-rank tie handling
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return float(ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0), float(n_pos * n_neg)


def ts_auc(pred, y_true):
    """Time-Stratified AUC (the competition metric). For each online step t (n_online),
    take the AUC across the series alive at t, averaged with weight w(t)=n_pos*n_neg —
    i.e. pool only *within-step* pos/neg pairs. A series-constant score -> 0.5."""
    pred = pred.reindex(y_true.index)
    t = y_true.groupby(level="id").cumcount().to_numpy()     # 0-based online step index
    y = y_true.to_numpy().astype(int); p = pred.to_numpy()
    order = np.argsort(t, kind="mergesort")
    t, y, p = t[order], y[order], p[order]
    num = den = 0.0; i = 0
    while i < len(t):                                         # one stratum per step t
        j = i
        while j + 1 < len(t) and t[j + 1] == t[i]:
            j += 1
        c, w = _auc_pair_stat(p[i:j + 1], y[i:j + 1]); num += c; den += w; i = j + 1
    return num / den


def pooled_auc(pred, y_true):
    """Global pooled ROC AUC over all online rows (diagnostic only; NOT the metric)."""
    c, w = _auc_pair_stat(pred.reindex(y_true.index).to_numpy(), y_true.to_numpy())
    return c / w


def score_predictions(pred, y_true):
    """Score (id, time) online predictions with the competition metric (TS-AUC)."""
    pred = pred.reindex(y_true.index)
    assert pred.notna().all(), "predictions missing for some (id, time) online rows"
    return ts_auc(pred, y_true)

In [ ]:
# sanity check (TS-AUC): random ~ 0.5, perfect = 1.0
y_red = pd.read_parquet("y_test.reduced.parquet")["target"]
rng = np.random.default_rng(0)
rand_pred = pd.Series(rng.random(len(y_red)), index=y_red.index)
print("random  TS-AUC:", round(score_predictions(rand_pred, y_red), 4))
print("perfect TS-AUC:", round(score_predictions(y_red.astype(float), y_red), 4))

### Worked example — how TS-AUC is computed

4 tiny series, online segments 2–3 steps long. **Step `t` = the t-th online observation** (aligned across series, not absolute time); a series is "alive" at `t` if it has that many online obs. Each cell is `target` (1 = break already happened) and the model `score`.

| series | step 1 | step 2 | step 3 | notes |
|---|---|---|---|---|
| **A** | y=1, s=.80 | y=1, s=.90 | y=1, s=.95 | breaks at step 1 |
| **B** | y=0, s=.20 | y=0, s=.30 | y=0, s=.25 | never breaks |
| **C** | y=0, s=.40 | y=1, s=.55 | y=1, s=.70 | breaks at step 2 |
| **D** | y=0, s=.50 | y=0, s=.65 | — | never breaks; only 2 obs (false alarm at step 2) |

**One AUC per step, across series** (weight `w(t) = n_pos·n_neg` = number of pos/neg pairs):

- **Step 1** — pos {A=.80}, neg {B=.20, C=.40, D=.50}: 1×3 = 3 pairs, all correct → **AUC=1.0**, `w=3`.
- **Step 2** — pos {A=.90, C=.55}, neg {B=.30, D=.65}: 2×2 = 4 pairs; A>.30 ✓, A>.65 ✓, C>.30 ✓, C=.55 vs D=.65 ✗ → 3/4 → **AUC=0.75**, `w=4`.
- **Step 3** — alive A,B,C (D has no step 3); pos {A=.95, C=.70}, neg {B=.25}: 2×1 = 2 pairs, both correct → **AUC=1.0**, `w=2`.

**Combine** (pair-count weighted average — i.e. total correct pairs / total pairs):

$$\text{TS-AUC}=\frac{\sum_t w(t)\,\text{AUC}(t)}{\sum_t w(t)}=\frac{3(1.0)+4(0.75)+2(1.0)}{3+4+2}=\frac{8}{9}\approx 0.889$$

Steps with only one class have `w=0` and drop out. Each step's AUC is the rank-sum in `_auc_pair_stat` (e.g. step 2 sorted `B .30, C .55, D .65, A .90`; positives C,A at ranks 2,4 → `(6 − 2·3/2)/(2·2) = 3/4`).

**Why `n_online` → 0.5 here:** if the score were the step number, then at step 2 *every* alive series scores "2" — identical — so all pairs tie and AUC(2)=0.5; same every step → TS-AUC=0.5. The metric can't see a score that doesn't vary *between series at the same step*. (A global pooled AUC would instead also compare, e.g., A's step-3 .95 against D's step-1 .50 — a cross-step pair TS-AUC never forms.)

## Step 2 — Baselines to beat

Under **TS-AUC** the only way to score is to tell series apart *at the same online step*, so:

- **The time prior is worthless.** `n_online` — and any score identical across series at a step — is **exactly 0.5**. (Under a naive *pooled* AUC it looks like ~0.68, but that's an artifact of comparing across steps, not skill. Ignore it.)
- **The real baseline is a value detector** — a causal accumulator comparing the online prefix to the series' *own* reference via expanding mean-shift z and |log variance-ratio|. This actually detects a change, and reaches only **~0.55** — so genuine detection is the whole game.

Since comparisons are cross-sectional at each step, features must be **comparable across series** — i.e. normalized by each series' own reference stats (as ours are).

In [12]:
VAR_MIN = 20   # min online points before trusting a variance-ratio signal


def _ref_stats(X):
    """Per-series reference-segment (period==1) mean / std / var."""
    ref = X[X["period"] == 1]["value"]
    rs = ref.groupby(level="id").agg(rmean="mean", rvar=lambda s: s.var(ddof=0))
    rs["rstd"] = np.sqrt(rs["rvar"]) + 1e-9
    rs["rvar"] = rs["rvar"] + 1e-9
    return rs


def causal_baseline(X):
    """Value-based causal detector: expanding mean-shift z + |log variance-ratio|
    (variance guarded until VAR_MIN points). Cumulative stats only -> causal.
    Returns scores indexed by (id, time) over the online rows."""
    rs = _ref_stats(X)
    onl = X[X["period"] == 2][["value"]].copy()
    ids = onl.index.get_level_values("id")
    for c in ("rmean", "rstd", "rvar"):
        onl[c] = rs[c].reindex(ids).values
    g = onl.groupby(level="id")
    onl["n"] = g.cumcount() + 1
    onl["exp_mean"] = g["value"].cumsum() / onl["n"]
    onl["sq"] = onl["value"] ** 2
    onl["exp_var"] = (g["sq"].cumsum() / onl["n"] - onl["exp_mean"] ** 2).clip(lower=0)
    z  = np.abs((onl["exp_mean"] - onl["rmean"]) / (onl["rstd"] / np.sqrt(onl["n"])))
    lv = np.abs(np.where(onl["n"] >= VAR_MIN,
                         np.log((onl["exp_var"] + 1e-9) / onl["rvar"]), 0.0))
    return pd.Series(z.values + 2.0 * lv, index=onl.index)

In [ ]:
X_red = pd.read_parquet("X_test.reduced.parquet")
n_online = X_red[X_red["period"] == 2].groupby(level="id").cumcount() + 1   # elapsed online steps
base = causal_baseline(X_red)
print("n_online       TS-AUC: %.4f   (pooled: %.4f)  <- time prior, neutralized by TS-AUC"
      % (score_predictions(n_online, y_red), pooled_auc(n_online, y_red)))
print("value detector TS-AUC: %.4f   (pooled: %.4f)"
      % (score_predictions(base, y_red), pooled_auc(base, y_red)))

## Step 3 — Causal feature builder

One feature row per online timestep, all causal (cumulative or trailing-rolling only). Reusable for both train and inference. Followed by a **leakage test**: features computed on a truncated prefix must equal the full-series features restricted to that prefix.

In [14]:
def build_features(X, roll=30):
    """Per-online-timestep causal feature matrix, indexed by (id, time) over online rows.
    Every feature at time t uses only the reference segment + online values up to t
    (cumulative / trailing-rolling only), so it is safe for sequential inference."""
    rs = _ref_stats(X)
    onl = X[X["period"] == 2][["value"]].copy()
    ids = onl.index.get_level_values("id")
    for c in ("rmean", "rstd", "rvar"):
        onl[c] = rs[c].reindex(ids).values
    g = onl.groupby(level="id")
    onl["n"] = g.cumcount() + 1
    onl["exp_mean"] = g["value"].cumsum() / onl["n"]
    onl["sq"] = onl["value"] ** 2
    onl["exp_var"] = (g["sq"].cumsum() / onl["n"] - onl["exp_mean"] ** 2).clip(lower=0)
    roll_mean = g["value"].rolling(roll, min_periods=1).mean().reset_index(level=0, drop=True)
    roll_std  = g["value"].rolling(roll, min_periods=2).std().reset_index(level=0, drop=True)

    f = pd.DataFrame(index=onl.index)
    f["n_online"]         = onl["n"]                                                   # time prior
    f["last_z"]           = (onl["value"] - onl["rmean"]) / onl["rstd"]                # latest value vs reference
    f["exp_mean_shift_z"] = (onl["exp_mean"] - onl["rmean"]) / (onl["rstd"] / np.sqrt(onl["n"]))
    f["exp_logvar"]       = np.where(onl["n"] >= VAR_MIN,
                                     np.log((onl["exp_var"] + 1e-9) / onl["rvar"]), 0.0)
    f["roll_mean_shift"]  = (roll_mean - onl["rmean"]) / onl["rstd"]                   # recent-window shift
    f["roll_logstd"]      = np.log((roll_std.fillna(onl["rstd"]) + 1e-9) / onl["rstd"])
    f["evidence_cummax"]  = ((f["exp_mean_shift_z"].abs() + 2 * f["exp_logvar"].abs())
                             .groupby(level="id").cummax())                            # monotone accumulator
    return f

In [15]:
# leakage / causality test: features on a truncated prefix must equal the
# full-series features restricted to that prefix (no look-ahead).
sid = X_red.index.get_level_values("id")[0]
Xs = X_red.loc[[sid]]
full = build_features(Xs)
onl_times = Xs.loc[sid].query("period == 2").index
cut = onl_times[len(onl_times) // 2]                      # truncate halfway through the online segment
Xcut = Xs[Xs.index.get_level_values("time") <= cut]
prefix = build_features(Xcut)
pd.testing.assert_frame_equal(full.loc[prefix.index], prefix, check_like=True)
print("causality OK: prefix features identical up to t =", cut, "| rows checked:", len(prefix))

causality OK: prefix features identical up to t = 3470 | rows checked: 159


In [ ]:
feat_red = build_features(X_red)
print("feature matrix shape:", feat_red.shape, "| any NaN:", bool(feat_red.isna().any().any()))
print("\nper-feature AUC (reduced set):")
print(f"  {'feature':18s} {'TS-AUC':>7s} {'pooled':>7s}")
for c in feat_red.columns:
    print(f"  {c:18s} {score_predictions(feat_red[c], y_red):7.4f} {pooled_auc(feat_red[c], y_red):7.4f}")
feat_red.head()

## Step 4 — model (GroupKFold OOF)

Train LightGBM on the causal features with 5-fold **GroupKFold by `id`** (no series split across folds), then score the out-of-fold predictions with **TS-AUC**. The model trains on binary log-loss; only the *evaluation* uses TS-AUC. The target to beat is the **value detector (~0.55)** — `n_online` (~0.5) is not a target and can even be dropped, since it shifts every series equally at a step and is TS-AUC-neutral. Needs the `structural-break` env (`lightgbm`, `scikit-learn`).

In [17]:
import time

t0 = time.time()
X_tr = pd.read_parquet("X_train.parquet")
F = build_features(X_tr)                                       # ~5M online rows; ~1-2 min (groupby rolling)
y = pd.read_parquet("y_train.parquet")["target"].reindex(F.index).astype("int8")
assert not F.isna().any().any() and y.notna().all()
print("features:", F.shape, "| target rate:", round(float(y.mean()), 4),
      "| built in %.0fs" % (time.time() - t0))

features: (5036517, 7) | target rate: 0.2557 | built in 6s


In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

feat_cols = F.columns.tolist()
Xmat   = F.to_numpy(dtype=np.float32)
yvec   = y.to_numpy()
groups = F.index.get_level_values("id").to_numpy()

params = dict(objective="binary", n_estimators=500, learning_rate=0.03,
              num_leaves=63, min_child_samples=300, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, verbosity=-1)

oof = np.zeros(len(F))
importance = np.zeros(len(feat_cols))
for k, (tr, va) in enumerate(GroupKFold(n_splits=5).split(Xmat, yvec, groups)):
    m = lgb.LGBMClassifier(**params).fit(Xmat[tr], yvec[tr])
    oof[va] = m.predict_proba(Xmat[va])[:, 1]
    importance += m.feature_importances_
    fold_auc = ts_auc(pd.Series(oof[va], index=F.index[va]), y.iloc[va])   # complete series per fold
    print(f"fold {k}: TS-AUC = {fold_auc:.4f}")
oof_pred = pd.Series(oof, index=F.index)

In [ ]:
print("=== TS-AUC (train, out-of-fold) ===")
print("n_online (time prior) :", round(score_predictions(F["n_online"], y), 4), " <- ~0.5 by design")
print("value detector        :", round(score_predictions(causal_baseline(X_tr), y), 4))
print("LightGBM (all feats)  :", round(score_predictions(oof_pred, y), 4))
print("\n(pooled AUC, diagnostic only — inflated by the time prior)")
print("  n_online:", round(pooled_auc(F["n_online"], y), 4),
      "| LightGBM:", round(pooled_auc(oof_pred, y), 4))
print("\nmean feature importance (split count):")
print(pd.Series(importance / 5, index=feat_cols).sort_values(ascending=False).round(1))

### Read-out & next

- **Under TS-AUC, `n_online` ≈ 0.5** — the time prior is free to everyone, so it can't score. Real signal = the value detector (~0.55) and the model's lift above it. (The earlier "0.68 baseline" was pooled AUC leaking cross-step comparisons — ignore it.)
- Because scoring is cross-sectional at each step, **features must discriminate between series at the same `t`** → keep everything normalized by each series' own reference. `n_online` can be dropped from the model (TS-AUC-neutral).
- **Add detection features to `build_features`** — the gains are here: expanding **KS / Cramér–von Mises** vs reference, lag-1 autocorrelation shift, skew/kurtosis, short-vs-long window contrasts. Each change = one `score_predictions` (TS-AUC) number.
- Re-run the **leakage test** after adding any feature.
- **Step 5 (submission):** `train()` (fit on all rows) + sequential `infer()` holding streaming state (running sums, length-`roll` ring buffer) that recomputes these features causally per online tick.